<h1><img src="../../icons/tk_full_logo.svg" width="80" /> Zebra-Puzzle GRPO Fine-tune (Gemma 4 E2B)</h1>

Copyright (c) 2026 Thinkube Contributors. Licensed under Apache-2.0.
SPDX-License-Identifier: Apache-2.0

This notebook uses the Unsloth library (Apache-2.0) and the TRL library (Apache-2.0).
It is an independent work and is NOT derived from any LGPL-3.0 notebook in the
unslothai/notebooks repository.

---

## What This Notebook Does

Fine-tunes **Gemma 4 E2B** on zebra logic puzzles using **GRPO** (Group Relative Policy Optimization),
demonstrating measurable before/after improvement in logical reasoning.

| Stage | What Happens |
|-------|-------------|
| **Baseline** | Evaluate the untrained model on 200 held-out puzzles |
| **Training** | GRPO with a verifiable reward function (fraction of correct cell assignments) |
| **Evaluation** | Re-evaluate on the same 200 puzzles, compare mean reward |
| **Export** | Save LoRA adapter for deployment |

**Target hardware:** NVIDIA DGX Spark (Blackwell, unified memory, ~9 GB for E2B GRPO)

## [Optional] Install Dependencies

Skip this cell if running in an environment that already has Unsloth and TRL installed
(e.g. Unsloth's official container or thinkube's `fine-tuning` venv).

In [ ]:
# Uncomment to install dependencies:
# !pip install unsloth trl datasets python-constraint matplotlib

## Imports

In [ ]:
import json
import random
import time

import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

import zebra_dataset

## Configuration

All training-relevant numbers live here. Set `SMOKE_TEST = True` to verify the
pipeline end-to-end in a couple of minutes before committing to a full run.

In [ ]:
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LEN = 2048
LORA_RANK = 32
LORA_ALPHA = 32

NUM_TRAIN_PUZZLES = 2000
NUM_EVAL_PUZZLES = 200
TRAIN_SEED = 1
EVAL_SEED = 99
SEED = 42

NUM_GENERATIONS = 8
LEARNING_RATE = 5e-6
MAX_STEPS = 300
GRADIENT_ACCUMULATION = 4
KL_BETA = 0.04
MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = 1024

OUTPUT_DIR = "zebra_lora_adapter"

# Quick pipeline check — do NOT interpret reward deltas as meaningful
SMOKE_TEST = False

if SMOKE_TEST:
    NUM_TRAIN_PUZZLES = 100
    NUM_EVAL_PUZZLES = 50
    MAX_STEPS = 10
    print("SMOKE TEST mode: reduced dataset and steps")

print(f"Model: {MODEL_NAME}")
print(f"Train puzzles: {NUM_TRAIN_PUZZLES}, Eval puzzles: {NUM_EVAL_PUZZLES}")
print(f"LoRA rank: {LORA_RANK}, LR: {LEARNING_RATE}, Steps: {MAX_STEPS}")

## Load Model & Tokenizer

Load Gemma 4 E2B with LoRA adapters. We use bf16 precision (required by Gemma 4)
and disable fast inference (GRPO needs the standard generation path).

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    dtype=torch.bfloat16,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.2f}%)")

## Generate Dataset

Generate training and evaluation puzzle sets with disjoint seeds.
Each puzzle is wrapped as a single-turn chat using the model's template.

In [ ]:
SYSTEM_PROMPT = (
    "You are a logic puzzle solver. Reason step by step through the clues, "
    "then write the final answer with exactly one line per house in this format:\n"
    "House 1: category1=value | category2=value | ...\n"
    "House 2: category1=value | category2=value | ...\n"
    "(and so on for all houses)"
)

def make_puzzles(num_puzzles, seed):
    """Generate puzzles using zebra_dataset with a fixed RNG."""
    rng = random.Random(seed)
    puzzles = []
    attempts = 0
    while len(puzzles) < num_puzzles:
        theme = rng.choice(zebra_dataset.THEMES)
        puzzle = zebra_dataset.generate(theme, 5, rng)
        attempts += 1
        if puzzle is not None:
            puzzles.append(puzzle)
        if len(puzzles) % 100 == 0 and len(puzzles) > 0:
            print(f"  generated {len(puzzles)}/{num_puzzles}...")
    print(f"  done: {len(puzzles)} puzzles in {attempts} attempts")
    return puzzles

print("Generating training puzzles...")
train_puzzles = make_puzzles(NUM_TRAIN_PUZZLES, TRAIN_SEED)

print("Generating evaluation puzzles...")
eval_puzzles = make_puzzles(NUM_EVAL_PUZZLES, EVAL_SEED)

# Verify no overlap between train and eval
train_sigs = {json.dumps(p["solution"], sort_keys=True) for p in train_puzzles}
eval_sigs = {json.dumps(p["solution"], sort_keys=True) for p in eval_puzzles}
assert train_sigs.isdisjoint(eval_sigs), "train/eval overlap detected — check seeds"
print(f"\nTrain/eval disjoint: confirmed ({len(train_sigs)} vs {len(eval_sigs)} unique solutions)")

In [ ]:
def puzzle_to_chat(puzzle):
    """Convert a puzzle dict to a chat-formatted training row."""
    prompt_text = zebra_dataset.format_prompt(puzzle)
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ],
        # Pass puzzle data through for the reward function
        "puzzle_solution": json.dumps(puzzle["solution"]),
        "puzzle_categories": json.dumps(list(puzzle["categories"].keys())),
        "puzzle_n": puzzle["N"],
    }

train_rows = [puzzle_to_chat(p) for p in train_puzzles]
train_dataset = Dataset.from_list(train_rows)

print(f"Training dataset: {len(train_dataset)} rows")
print(f"Sample prompt (first 200 chars):\n{train_rows[0]['prompt'][1]['content'][:200]}...")

## Baseline Evaluation

Run the untrained model on the evaluation set to establish a baseline.
If the baseline is already strong (≥ 0.45), the puzzles may be too easy.

In [ ]:
def evaluate_model(model, tokenizer, puzzles, max_new_tokens=MAX_COMPLETION_LENGTH, temperature=0.7):
    """Generate answers for puzzles and compute rewards."""
    FastLanguageModel.for_inference(model)
    rewards = []
    sample_outputs = []

    for idx, puzzle in enumerate(puzzles):
        prompt_text = zebra_dataset.format_prompt(puzzle)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
            )

        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        score = zebra_dataset.reward(generated, puzzle)
        rewards.append(score)

        if len(sample_outputs) < 2:
            sample_outputs.append({
                "puzzle_idx": idx,
                "reward": score,
                "expected": zebra_dataset.format_expected(puzzle),
                "generated": generated[:800],
            })

        if (idx + 1) % 50 == 0:
            print(f"  evaluated {idx + 1}/{len(puzzles)}, running mean: {sum(rewards)/len(rewards):.3f}")

    FastLanguageModel.for_training(model)
    return rewards, sample_outputs


# Quick sanity check on first 10 puzzles
print("Sanity check on 10 eval puzzles...")
quick_rewards, _ = evaluate_model(model, tokenizer, eval_puzzles[:10])
quick_mean = sum(quick_rewards) / len(quick_rewards)
print(f"Quick baseline: {quick_mean:.3f}")

if quick_mean >= 0.45:
    print("WARNING: Baseline is already strong (>= 0.45). Consider harder puzzles.")
    print("  Options: increase N to 6, reduce clue minimisation, or exclude cl_position.")

In [ ]:
print("Running full baseline evaluation...")
baseline_rewards, baseline_samples = evaluate_model(model, tokenizer, eval_puzzles)
baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
baseline_perfect = sum(1 for r in baseline_rewards if r >= 1.25) / len(baseline_rewards)

print(f"\nBaseline Results:")
print(f"  Mean reward: {baseline_mean:.3f}")
print(f"  % perfect:   {100*baseline_perfect:.1f}%")

print(f"\n--- Sample Generation ---")
for s in baseline_samples:
    print(f"\nPuzzle {s['puzzle_idx']} (reward={s['reward']:.3f}):")
    print(f"Expected:\n{s['expected']}")
    print(f"Generated:\n{s['generated'][:500]}")

## Reward Wrapper

Wraps `zebra_dataset.reward()` in the signature expected by TRL's `GRPOTrainer`.
Adds a small format bonus when the output contains exactly N `House N:` lines.

In [ ]:
import re

def zebra_reward_func(completions, puzzle_solution, puzzle_categories, puzzle_n, **kwargs):
    """Compute rewards for a batch of GRPO completions.

    Args:
        completions: list of [{"content": str}] — one per generation
        puzzle_solution: JSON-encoded solution dict (passed through dataset)
        puzzle_categories: JSON-encoded category list
        puzzle_n: number of houses
    """
    rewards = []
    for completion, sol_json, cats_json, n_val in zip(
        completions, puzzle_solution, puzzle_categories, puzzle_n
    ):
        generated_text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        n = int(n_val)

        # Reconstruct puzzle dict for the reward function
        puzzle = {
            "solution": json.loads(sol_json),
            "categories": {c: [] for c in json.loads(cats_json)},
            "N": n,
        }

        # Core reward: fraction of correct assignments + perfect bonus
        score = zebra_dataset.reward(generated_text, puzzle)

        # Format bonus: small reward for correct number of House N: lines
        house_lines = re.findall(r"^\s*House\s+\d+:", generated_text, re.MULTILINE)
        if len(house_lines) == n:
            score += 0.05

        rewards.append(float(score))

    return rewards

print("Reward function ready.")

## GRPO Trainer Setup

In [ ]:
training_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS,
    max_grad_norm=0.1,
    beta=KL_BETA,
    logging_steps=1,
    save_steps=MAX_STEPS,
    report_to="none",
    output_dir=OUTPUT_DIR,
    seed=SEED,
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[zebra_reward_func],
    args=training_config,
    train_dataset=train_dataset,
)

print(f"Trainer ready: {MAX_STEPS} steps, {NUM_GENERATIONS} generations/prompt")

## Training

Run GRPO training. Rewards may not increase for the first 100-150 steps — be patient.

In [ ]:
start_time = time.time()

try:
    train_result = trainer.train()
except KeyboardInterrupt:
    print("\nTraining interrupted — model state is recoverable.")
    train_result = None

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")

# Plot loss curve
try:
    import matplotlib.pyplot as plt
    log_history = trainer.state.log_history
    losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e]
    if losses:
        steps, loss_vals = zip(*losses)
        plt.figure(figsize=(10, 4))
        plt.plot(steps, loss_vals, alpha=0.7)
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("GRPO Training Loss")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except ImportError:
    losses = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
    if losses:
        print("Last 10 losses:")
        for step, loss in losses[-10:]:
            print(f"  step {step}: {loss:.4f}")

## Post-Training Evaluation

Re-evaluate on the same held-out puzzles to measure improvement.

In [ ]:
print("Running post-training evaluation...")
trained_rewards, trained_samples = evaluate_model(model, tokenizer, eval_puzzles)
trained_mean = sum(trained_rewards) / len(trained_rewards)
trained_perfect = sum(1 for r in trained_rewards if r >= 1.25) / len(trained_rewards)

delta_mean = trained_mean - baseline_mean
delta_perfect = trained_perfect - baseline_perfect

print(f"\n{'':20s} {'baseline':>10s} {'trained':>10s} {'delta':>10s}")
print(f"{'─'*52}")
print(f"{'mean reward':20s} {baseline_mean:10.3f} {trained_mean:10.3f} {delta_mean:+10.3f}")
print(f"{'% perfect':20s} {100*baseline_perfect:9.1f}% {100*trained_perfect:9.1f}% {100*delta_perfect:+9.1f}%")

if delta_mean >= 0.15:
    print(f"\n*** Target met: +{delta_mean:.3f} >= +0.15 ***")
else:
    print(f"\nTarget NOT met (+{delta_mean:.3f} < +0.15). See spec section 9 for tuning guidance.")

## Qualitative Comparison

Side-by-side: base model vs trained model on the same held-out puzzle.

In [ ]:
from IPython.display import display, Markdown

comparison_puzzle = eval_puzzles[0]
expected = zebra_dataset.format_expected(comparison_puzzle)

base_gen = baseline_samples[0]["generated"] if baseline_samples else "(no baseline sample)"
trained_gen = trained_samples[0]["generated"] if trained_samples else "(no trained sample)"

output = f"""
## Puzzle

```
{zebra_dataset.format_prompt(comparison_puzzle)[:600]}
```

## Expected Answer

```
{expected}
```

## Base Model (reward={baseline_samples[0]['reward']:.3f})

```
{base_gen[:600]}
```

## Trained Model (reward={trained_samples[0]['reward']:.3f})

```
{trained_gen[:600]}
```
"""

display(Markdown(output))

## Save LoRA Adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
abs_path = os.path.abspath(OUTPUT_DIR)
print(f"LoRA adapter saved to: {abs_path}")
print(f"Contents: {os.listdir(abs_path)}")

## [Optional] Export to GGUF

GGUF export from Unsloth is best-effort for very new model architectures.

In [ ]:
# Uncomment to export:
# model.save_pretrained_gguf("zebra_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF export complete.")